In [1]:
import pandas as pd
import glob

path = './data'
all_files = glob.glob(path + '**/*.parquet', recursive=True)

dfs = []
for f in all_files:
    temp = pd.read_parquet(f)
    temp['source_file'] = f.split('/')[-1]
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel distribution:")
print(df['Label'].value_counts())

Shape: (2313810, 79)

Columns: ['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count'

In [2]:
df['target'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

print(df['target'].value_counts())
print(f"\nImbalance ratio: {df['target'].value_counts()[0] / df['target'].value_counts()[1]:.2f}:1")

target
0    1977318
1     336492
Name: count, dtype: int64

Imbalance ratio: 5.88:1


In [3]:
import numpy as np

print("Null values:", df.isnull().sum().sum())
print("Infinite values:", np.isinf(df.select_dtypes(include=np.number)).sum().sum())

print("\nColumns with nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nColumns with infinite values:")
numeric_cols = df.select_dtypes(include=np.number).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
print(inf_counts[inf_counts > 0])

Null values: 0
Infinite values: 0

Columns with nulls:
Series([], dtype: int64)

Columns with infinite values:
Series([], dtype: int64)


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features and target
drop_cols = ['Label', 'source_file', 'target']
X = df.drop(columns=drop_cols)
y = df['target']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTrain label distribution:")
print(y_train.value_counts())
print("\nTest label distribution:")
print(y_test.value_counts())

Train size: (1851048, 77)
Test size: (462762, 77)

Train label distribution:
target
0    1581854
1     269194
Name: count, dtype: int64

Test label distribution:
target
0    395464
1     67298
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("After SMOTE:")
print("Train size:", X_train_resampled.shape)
import pandas as pd
print(pd.Series(y_train_resampled).value_counts())

c:\Users\User\anaconda3\Lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


After SMOTE:
Train size: (3163708, 77)
target
0    1581854
1    1581854
Name: count, dtype: int64


In [6]:
from sklearn.ensemble import RandomForestClassifier
import time

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1  # uses all available CPU cores
)

start = time.time()
rf.fit(X_train_resampled, y_train_resampled)
end = time.time()

print(f"Training time: {(end - start):.2f} seconds")

Training time: 587.69 seconds


In [7]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# Predict on test set
y_pred = rf.predict(X_test_scaled)
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN)
FNR = FN / (FN + TP)

print(f"\nFalse Positive Rate (FPR): {FPR:.4f}")
print(f"False Negative Rate (FNR): {FNR:.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00    395464
      Attack       0.99      1.00      1.00     67298

    accuracy                           1.00    462762
   macro avg       1.00      1.00      1.00    462762
weighted avg       1.00      1.00      1.00    462762


Confusion Matrix:
[[395124    340]
 [   259  67039]]

False Positive Rate (FPR): 0.0009
False Negative Rate (FNR): 0.0038
AUC-ROC: 1.0000


In [8]:
import pandas as pd

feature_names = X.columns.tolist()
importances = rf.feature_importances_

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print(feat_imp.head(20))

                     feature  importance
12    Bwd Packet Length Mean    0.068322
13     Bwd Packet Length Std    0.065146
54      Avg Bwd Segment Size    0.064666
41         Packet Length Std    0.062851
10     Bwd Packet Length Max    0.059497
42    Packet Length Variance    0.054527
52           Avg Packet Size    0.042083
5   Bwd Packets Length Total    0.041410
37             Bwd Packets/s    0.039159
40        Packet Length Mean    0.035799
11     Bwd Packet Length Min    0.033607
21              Fwd IAT Mean    0.032448
39         Packet Length Max    0.030512
20             Fwd IAT Total    0.024199
34         Fwd Header Length    0.023763
6      Fwd Packet Length Max    0.020218
2          Total Fwd Packets    0.020104
17              Flow IAT Std    0.018257
61       Subflow Fwd Packets    0.016965
23               Fwd IAT Max    0.016612


In [9]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib
import numpy as np
import pandas as pd

# Evaluate
y_pred = rf.predict(X_test_scaled)
y_pred_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Classification report
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()
FPR = FP / (FP + TN)
FNR = FN / (FN + TP)
print(f"FPR: {FPR:.4f}")
print(f"FNR: {FNR:.4f}")
print(f"AUC-ROC: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Feature importance
feature_names = X.columns.tolist()
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("\nTop 20 features:")
print(feat_imp.head(20))

              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00    395464
      Attack       0.99      1.00      1.00     67298

    accuracy                           1.00    462762
   macro avg       1.00      1.00      1.00    462762
weighted avg       1.00      1.00      1.00    462762

FPR: 0.0009
FNR: 0.0038
AUC-ROC: 1.0000

Top 20 features:
                     feature  importance
12    Bwd Packet Length Mean    0.068322
13     Bwd Packet Length Std    0.065146
54      Avg Bwd Segment Size    0.064666
41         Packet Length Std    0.062851
10     Bwd Packet Length Max    0.059497
42    Packet Length Variance    0.054527
52           Avg Packet Size    0.042083
5   Bwd Packets Length Total    0.041410
37             Bwd Packets/s    0.039159
40        Packet Length Mean    0.035799
11     Bwd Packet Length Min    0.033607
21              Fwd IAT Mean    0.032448
39         Packet Length Max    0.030512
20             Fwd IAT Total    0.024199


In [10]:
# Save
from sklearn.pipeline import Pipeline
import joblib

rf_pipeline = Pipeline([
    ("scaler", scaler),
    ("rf", rf)
])

joblib.dump(rf_pipeline, "./training_model/rf_pipeline.pkl")
np.save('./training_model/X_test_scaled.npy', X_test_scaled)
np.save('./training_model/y_test.npy', np.array(y_test))
feat_imp.to_csv('./training_model/feature_importance.csv', index=False)
print("\nAll saved.")


All saved.


In [11]:
from sklearn.model_selection import cross_val_score

X_cv_sample = X_train_resampled[:50000]
y_cv_sample = y_train_resampled[:50000]

cv_scores = cross_val_score(rf, X_cv_sample, y_cv_sample,
                             cv=5, scoring='f1', n_jobs=-1)
print("CV F1 scores:", cv_scores)
print("Mean:", cv_scores.mean())
print("Std:", cv_scores.std())

CV F1 scores: [0.9925017  0.98871795 0.99008547 0.98943782 0.98913043]
Mean: 0.9899746754329914
Std: 0.00134003494899782


In [12]:
for i, col in enumerate(X.columns.tolist()):
    print(i, col)

0 Protocol
1 Flow Duration
2 Total Fwd Packets
3 Total Backward Packets
4 Fwd Packets Length Total
5 Bwd Packets Length Total
6 Fwd Packet Length Max
7 Fwd Packet Length Min
8 Fwd Packet Length Mean
9 Fwd Packet Length Std
10 Bwd Packet Length Max
11 Bwd Packet Length Min
12 Bwd Packet Length Mean
13 Bwd Packet Length Std
14 Flow Bytes/s
15 Flow Packets/s
16 Flow IAT Mean
17 Flow IAT Std
18 Flow IAT Max
19 Flow IAT Min
20 Fwd IAT Total
21 Fwd IAT Mean
22 Fwd IAT Std
23 Fwd IAT Max
24 Fwd IAT Min
25 Bwd IAT Total
26 Bwd IAT Mean
27 Bwd IAT Std
28 Bwd IAT Max
29 Bwd IAT Min
30 Fwd PSH Flags
31 Bwd PSH Flags
32 Fwd URG Flags
33 Bwd URG Flags
34 Fwd Header Length
35 Bwd Header Length
36 Fwd Packets/s
37 Bwd Packets/s
38 Packet Length Min
39 Packet Length Max
40 Packet Length Mean
41 Packet Length Std
42 Packet Length Variance
43 FIN Flag Count
44 SYN Flag Count
45 RST Flag Count
46 PSH Flag Count
47 ACK Flag Count
48 URG Flag Count
49 CWE Flag Count
50 ECE Flag Count
51 Down/Up Ratio
52 Av